## Lab Test Booking Assistant



In [16]:
%pip -q install -U langchain langchain-groq ddgs

Note: you may need to restart the kernel to use updated packages.


In [17]:
import os, re, sqlite3
from datetime import date, timedelta
from typing import Optional
from typing_extensions import TypedDict

from langchain_groq import ChatGroq
from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime
from langgraph.checkpoint.memory import InMemorySaver

try:
    from kaggle_secrets import UserSecretsClient
    GROQ_API_KEY = UserSecretsClient().get_secret("GROQ_API_KEY")
except Exception:
    GROQ_API_KEY = os.getenv("GROQ_API_KEY")

if not GROQ_API_KEY:
    raise ValueError("Add GROQ_API_KEY to Kaggle Secrets.")

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    api_key=GROQ_API_KEY
)
print("Groq model connected.")

Groq model connected.


In [18]:
DB_FILE = "lab_booking.db"

def connect_db():
    con = sqlite3.connect(DB_FILE)
    con.row_factory = sqlite3.Row
    con.execute("PRAGMA foreign_keys = ON")
    return con

con = connect_db()
con.executescript("""
CREATE TABLE IF NOT EXISTS tests(
    test_id INTEGER PRIMARY KEY AUTOINCREMENT,
    test_name TEXT UNIQUE,
    category TEXT,
    price INTEGER
);
CREATE TABLE IF NOT EXISTS slots(
    slot_id INTEGER PRIMARY KEY AUTOINCREMENT,
    test_id INTEGER,
    slot_date TEXT,
    slot_time TEXT,
    is_booked INTEGER DEFAULT 0,
    UNIQUE(test_id, slot_date, slot_time),
    FOREIGN KEY(test_id) REFERENCES tests(test_id)
);
CREATE TABLE IF NOT EXISTS bookings(
    booking_id INTEGER PRIMARY KEY AUTOINCREMENT,
    user_id TEXT,
    test_id INTEGER,
    slot_id INTEGER,
    status TEXT DEFAULT 'CONFIRMED',
    FOREIGN KEY(test_id) REFERENCES tests(test_id),
    FOREIGN KEY(slot_id) REFERENCES slots(slot_id)
);
CREATE UNIQUE INDEX IF NOT EXISTS one_active_booking
ON bookings(slot_id) WHERE status='CONFIRMED';
""")
con.close()
print("Database created.")

Database created.


In [19]:
sample_tests = [
    ("CBC", "Blood", 500),
    ("Blood Sugar", "Blood", 300),
    ("Lipid Profile", "Blood", 800),
    ("Thyroid Profile", "Hormone", 900),
    ("Vitamin D", "Vitamin", 1200),
    ("Vitamin B12", "Vitamin", 1000),
    ("Urine Routine", "Urine", 400),
    ("Kidney Function Test", "Blood", 700)
]

con = connect_db()
for name, category, price in sample_tests:
    con.execute(
        "INSERT OR IGNORE INTO tests(test_name,category,price) VALUES(?,?,?)",
        (name, category, price)
    )

for test in con.execute("SELECT test_id,test_name FROM tests").fetchall():
    for i, tm in enumerate(["09:00", "11:00", "14:00"]):
        day = (date.today() + timedelta(days=i+1)).isoformat()
        full = 1 if test["test_name"] == "Kidney Function Test" else 0
        con.execute(
            "INSERT OR IGNORE INTO slots(test_id,slot_date,slot_time,is_booked) VALUES(?,?,?,?)",
            (test["test_id"], day, tm, full)
        )
con.commit()
con.close()
print("8 tests and 24 slots are ready.")

8 tests and 24 slots are ready.


In [20]:
class UserContext(TypedDict):
    user_id: str
    approved_action: Optional[str]
    approved_test_id: Optional[int]
    approved_slot_id: Optional[int]
    approved_booking_id: Optional[int]

print("User ID is supplied by Python context, not by the model.")

User ID is supplied by Python context, not by the model.


In [21]:
@tool
def list_tests() -> str:
    """Show all lab tests and prices from SQLite."""
    con = connect_db()
    rows = con.execute(
        "SELECT test_id,test_name,category,price FROM tests ORDER BY test_id"
    ).fetchall()
    con.close()
    return "\\n".join(
        f"{r['test_id']}. {r['test_name']} - {r['category']} - Rs.{r['price']}"
        for r in rows
    )

@tool
def check_slots(test_name: str) -> str:
    """Show free slots for a test from SQLite."""
    con = connect_db()
    rows = con.execute(
        """SELECT s.slot_id,s.slot_date,s.slot_time
           FROM slots s JOIN tests t ON t.test_id=s.test_id
           WHERE lower(t.test_name)=lower(?) AND s.is_booked=0
           ORDER BY s.slot_date,s.slot_time""",
        (test_name,)
    ).fetchall()
    con.close()
    if not rows:
        return "No free slots found."
    return "\\n".join(
        f"Slot {r['slot_id']}: {r['slot_date']} at {r['slot_time']}" for r in rows
    )

@tool
def my_bookings(runtime: ToolRuntime[UserContext]) -> str:
    """Show bookings of the current signed-in user only."""
    user_id = runtime.context["user_id"]
    con = connect_db()
    rows = con.execute(
        """SELECT b.booking_id,t.test_name,s.slot_date,s.slot_time,b.status
           FROM bookings b JOIN tests t ON b.test_id=t.test_id
           JOIN slots s ON b.slot_id=s.slot_id
           WHERE b.user_id=? ORDER BY b.booking_id""",
        (user_id,)
    ).fetchall()
    con.close()
    if not rows:
        return "No bookings found."
    return "\\n".join(
        f"Booking {r['booking_id']}: {r['test_name']} {r['slot_date']} {r['slot_time']} ({r['status']})"
        for r in rows
    )

print("Read tools ready.")

Read tools ready.


In [22]:
def save_booking(user_id, test_id, slot_id):
    con = connect_db()
    try:
        con.execute("BEGIN IMMEDIATE")
        test = con.execute(
            "SELECT * FROM tests WHERE test_id=?", (test_id,)
        ).fetchone()
        slot = con.execute(
            "SELECT * FROM slots WHERE slot_id=? AND test_id=?",
            (slot_id, test_id)
        ).fetchone()

        if not test or not slot:
            raise ValueError("Test or slot does not exist.")
        if slot["is_booked"] == 1:
            raise ValueError("Slot is already booked.")

        duplicate = con.execute(
            "SELECT booking_id FROM bookings WHERE user_id=? AND test_id=? AND status='CONFIRMED'",
            (user_id, test_id)
        ).fetchone()
        if duplicate:
            raise ValueError("User already has this test booked.")

        con.execute(
            "INSERT INTO bookings(user_id,test_id,slot_id,status) VALUES(?,?,?,'CONFIRMED')",
            (user_id, test_id, slot_id)
        )
        con.execute("UPDATE slots SET is_booked=1 WHERE slot_id=?", (slot_id,))
        receipt = con.execute(
            """SELECT b.booking_id,b.user_id,t.test_name,t.price,
                      s.slot_date,s.slot_time,b.status
               FROM bookings b JOIN tests t ON b.test_id=t.test_id
               JOIN slots s ON b.slot_id=s.slot_id
               WHERE b.booking_id=last_insert_rowid()"""
        ).fetchone()
        con.commit()
        return "Receipt: " + str(dict(receipt))
    except Exception:
        con.rollback()
        raise
    finally:
        con.close()

def save_cancellation(user_id, booking_id):
    con = connect_db()
    try:
        con.execute("BEGIN IMMEDIATE")
        row = con.execute(
            "SELECT * FROM bookings WHERE booking_id=? AND user_id=? AND status='CONFIRMED'",
            (booking_id, user_id)
        ).fetchone()
        if not row:
            raise ValueError("Active booking not found for this user.")
        con.execute(
            "UPDATE bookings SET status='CANCELLED' WHERE booking_id=?",
            (booking_id,)
        )
        con.execute(
            "UPDATE slots SET is_booked=0 WHERE slot_id=?", (row["slot_id"],)
        )
        con.commit()
        return "Booking cancelled successfully."
    except Exception:
        con.rollback()
        raise
    finally:
        con.close()

print("Database write functions ready.")

Database write functions ready.


In [23]:
@tool
def book_test(test_id: int, slot_id: int, runtime: ToolRuntime[UserContext]) -> str:
    """Book a test only after approval for these exact IDs."""
    c = runtime.context
    if (c.get("approved_action") != "book"
        or c.get("approved_test_id") != test_id
        or c.get("approved_slot_id") != slot_id):
        return "Booking denied. Approval is required."
    try:
        return save_booking(c["user_id"], test_id, slot_id)
    except Exception as e:
        return "Booking failed: " + str(e)

@tool
def cancel_booking(booking_id: int, runtime: ToolRuntime[UserContext]) -> str:
    """Cancel a booking only after approval for this booking ID."""
    c = runtime.context
    if (c.get("approved_action") != "cancel"
        or c.get("approved_booking_id") != booking_id):
        return "Cancellation denied. Approval is required."
    try:
        return save_cancellation(c["user_id"], booking_id)
    except Exception as e:
        return "Cancellation failed: " + str(e)

print("Protected write tools ready.")

Protected write tools ready.


In [24]:
@tool
def general_info(question: str) -> str:
    """Search general educational information, not prices or availability."""
    from ddgs import DDGS
    results = DDGS().text(question, max_results=3)
    return "\\n".join(r.get("body", "") for r in results) or "No results."

In [25]:
SYSTEM_PROMPT = """
You are a fictional lab booking assistant.
Use list_tests and check_slots for prices and availability.
Use my_bookings for the current user only.
Use general_info only for general educational information.
Never diagnose or select tests based on symptoms.
Never guess prices or slots.
Only book or cancel when trusted context allows the exact action.
"""

tools = [list_tests, check_slots, my_bookings, book_test, cancel_booking, general_info]
memory = InMemorySaver()

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
    context_schema=UserContext,
    checkpointer=memory
)
print("Agent created.")

Agent created.


In [26]:
def chat(user_id, message, approved_action=None,
         approved_test_id=None, approved_slot_id=None,
         approved_booking_id=None):

    safe_message = re.sub(
        r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\\.[A-Za-z]{2,}",
        "[EMAIL REDACTED]",
        message
    )
    if safe_message != message:
        print("Email redacted before model received the message.")

    context = {
        "user_id": user_id,
        "approved_action": approved_action,
        "approved_test_id": approved_test_id,
        "approved_slot_id": approved_slot_id,
        "approved_booking_id": approved_booking_id
    }
    result = agent.invoke(
        {"messages": [{"role": "user", "content": safe_message}]},
        config={"configurable": {"thread_id": "lab_" + user_id}},
        context=context
    )
    for m in result["messages"]:
        if getattr(m, "type", "") == "tool":
            print("[TOOL]", m.name)
    print(result["messages"][-1].content)

In [27]:
def approve_booking(user_id, test_id, slot_id):
    con = connect_db()
    row = con.execute(
        """SELECT t.test_name,t.price,s.slot_date,s.slot_time,s.is_booked
           FROM tests t JOIN slots s ON t.test_id=s.test_id
           WHERE t.test_id=? AND s.slot_id=?""",
        (test_id, slot_id)
    ).fetchone()
    con.close()

    if not row or row["is_booked"]:
        print("Test or free slot not found.")
        return
    print("Booking details:", dict(row))
    answer = input("Approve booking? (yes/no): ").strip().lower()
    if answer == "yes":
        chat(user_id, f"Book test {test_id} in slot {slot_id}",
             "book", test_id, slot_id)
    else:
        print("Rejected. No database change.")

def approve_cancellation(user_id, booking_id):
    con = connect_db()
    row = con.execute(
        "SELECT * FROM bookings WHERE booking_id=? AND user_id=? AND status='CONFIRMED'",
        (booking_id, user_id)
    ).fetchone()
    con.close()
    if not row:
        print("Booking not found for this user.")
        return
    answer = input("Approve cancellation? (yes/no): ").strip().lower()
    if answer == "yes":
        chat(user_id, f"Cancel booking {booking_id}",
             "cancel", approved_booking_id=booking_id)
    else:
        print("Rejected. No database change.")

In [28]:
# Simulate one temporary failure, then retry the real database query.
retry_state = {"fail_once": True}

def slots_with_retry(test_name):
    for attempt in range(1, 3):
        try:
            if retry_state["fail_once"]:
                retry_state["fail_once"] = False
                raise RuntimeError("Temporary error for demonstration")
            print("Attempt", attempt, "succeeded.")
            return check_slots.invoke({"test_name": test_name})
        except Exception as error:
            print("Attempt", attempt, "failed:", error)
    return "Could not check slots."

print(slots_with_retry("CBC"))

Attempt 1 failed: Temporary error for demonstration
Attempt 2 succeeded.
Slot 4: 2026-11-21 at 09:00\nSlot 5: 2026-11-22 at 11:00\nSlot 6: 2026-11-23 at 14:00


In [29]:
def db_counts():
    con = connect_db()
    result = (
        con.execute("SELECT COUNT(*) FROM bookings").fetchone()[0],
        con.execute("SELECT COUNT(*) FROM slots WHERE is_booked=1").fetchone()[0]
    )
    con.close()
    return result

before = db_counts()
print("Before rejection:", before)
print("User selected NO. No write tool is called.")
after = db_counts()
print("After rejection:", after)
print("Database unchanged:", before == after)

# Each user has a separate conversation thread and booking query.
chat("USER_1", "Show my bookings.")
chat("USER_2", "Show my bookings.")

Before rejection: (0, 3)
User selected NO. No write tool is called.
After rejection: (0, 3)
Database unchanged: True
[TOOL] my_bookings
You currently don’t have any upcoming lab test bookings. If you’d like to schedule a test, just let me know which one you’re interested in, and I can check the available slots for you.
[TOOL] my_bookings
You currently don’t have any lab test bookings. If you’d like to schedule a test, just let me know which one you’re interested in!
